In [ ]:
import mesa
import numpy as np
import matplotlib.pyplot as plt
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Priority levels — externally assigned
class PriorityLevel(Enum):
    LOW = 1
    NORMAL = 2
    HIGH = 3
    CRITICAL = 4

# Aggression levels — internally generated
class AggressionLevel(Enum):
    PASSIVE = 1
    NORMAL = 2
    AGGRESSIVE = 3

# Priority coefficients
PRIORITY_COEFFICIENTS = {
    PriorityLevel.LOW: 0.3,
    PriorityLevel.NORMAL: 0.6,
    PriorityLevel.HIGH: 0.85,
    PriorityLevel.CRITICAL: 1.0
}

# Aggression coefficients
AGGRESSION_COEFFICIENTS = {
    AggressionLevel.PASSIVE: 0.3,
    AggressionLevel.NORMAL: 0.7,
    AggressionLevel.AGGRESSIVE: 1.0
}

# Weight balance between priority and aggression
# These are the key tuning parameters
W_PRIORITY = 0.6    # How much priority affects movement
W_AGGRESSION = 0.4  # How much aggression affects movement

print("Imports successful!")
print(f"Mesa version: {mesa.__version__}")
print("V7 Priority + Aggression Model: READY")
print(f"Priority weight:   {W_PRIORITY}")
print(f"Aggression weight: {W_AGGRESSION}")
print("\nFour Agent Quadrants:")
print("  Q1 — High Priority + High Aggression (emergency)")
print("  Q2 — High Priority + Low Aggression  (patient VIP)")
print("  Q3 — Low Priority  + Low Aggression  (background)")
print("  Q4 — Low Priority  + High Aggression (rogue)")

In [ ]:
class QueueAgent(mesa.Agent):
    def __init__(self, model, priority, aggression):
        super().__init__(model)
        self.priority = priority
        self.aggression = aggression
        self.priority_coeff = PRIORITY_COEFFICIENTS[priority]
        self.aggression_coeff = AGGRESSION_COEFFICIENTS[aggression]
        
        # Individual characteristics
        self.urgency = np.random.uniform(0.3, 1.0)
        self.social_inhibition = np.random.uniform(0.2, 0.8)
        self.exited = False
        self.entry_time = None
        self.exit_time = None
        self.latency = None
        
        # QoS tracking
        self.is_rogue = (
            self.aggression == AggressionLevel.AGGRESSIVE and
            self.priority == PriorityLevel.LOW
        )
        self.was_throttled = False
        
    @property
    def quadrant(self):
        high_p = self.priority in [
            PriorityLevel.HIGH, 
            PriorityLevel.CRITICAL
        ]
        high_a = self.aggression == AggressionLevel.AGGRESSIVE
        if high_p and high_a:
            return "Q1"
        elif high_p and not high_a:
            return "Q2"
        elif not high_p and not high_a:
            return "Q3"
        else:
            return "Q4"
    
    @property
    def base_behavior_score(self):
        """
        Two dimensional behavior score
        B = (P × w_p + A × w_a) × U / S
        """
        priority_component = self.priority_coeff * W_PRIORITY
        aggression_component = (
            self.aggression_coeff * W_AGGRESSION
        )
        base = ((priority_component + aggression_component) *
                self.urgency / self.social_inhibition)
        
        # Density modifier
        if self.pos:
            neighbors = self.model.grid.get_neighbors(
                self.pos,
                moore=True,
                include_center=False,
                radius=2
            )
            density_factor = 1.0 + (len(neighbors) * 0.05)
        else:
            density_factor = 1.0
            
        return base * density_factor
    
    @property
    def effective_behavior_score(self):
        """
        Behavior score after QoS filter applied
        Rogue agents get throttled
        """
        score = self.base_behavior_score
        if self.model.qos_enabled and self.is_rogue:
            score *= self.model.qos_throttle
            self.was_throttled = True
        return score
    
    def step(self):
        if self.exited:
            return
        if self.pos is None:
            return
        if self.entry_time is None:
            self.entry_time = self.model.steps
            
        x, y = self.pos
        move_prob = min(self.effective_behavior_score, 1.0)
        
        if np.random.random() < move_prob:
            new_y = y - 1
            
            if new_y < 0:
                self.model.grid.remove_agent(self)
                self.exited = True
                self.exit_time = self.model.steps
                if self.entry_time is not None:
                    self.latency = (self.exit_time -
                                   self.entry_time)
                self.model.exited_count += 1
                self.model.exit_times.append(self.model.steps)
                self.model.record_agent(self)
                return
            
            new_pos = (x, new_y)
            cell_contents = self.model.grid.get_cell_list_contents(
                [new_pos]
            )
            
            # Max occupancy based on aggression
            max_occupancy = (
                3 if self.aggression == AggressionLevel.AGGRESSIVE
                else 2 if self.aggression == AggressionLevel.NORMAL
                else 1
            )
            
            if len(cell_contents) < max_occupancy:
                self.model.grid.move_agent(self, new_pos)

print("Two dimensional agent class defined!")
print("Tracking: priority, aggression, quadrant")
print("QoS throttle: ENABLED")
print("Rogue detection: ENABLED")

In [ ]:
class BunchQueueModel(mesa.Model):
    def __init__(self, n_agents=100, width=20, height=30,
                 queue_scenario='mixed',
                 qos_enabled=False,
                 qos_throttle=0.3):
        super().__init__()
        
        self.width = width
        self.height = height
        self.steps = 0
        self.exited_count = 0
        self.exit_times = []
        self.total_agents = n_agents
        self.qos_enabled = qos_enabled
        self.qos_throttle = qos_throttle
        self.queue_scenario = queue_scenario
        
        # Results tracking by quadrant
        self.quadrant_latencies = {
            'Q1': [], 'Q2': [],
            'Q3': [], 'Q4': []
        }
        self.quadrant_counts = {
            'Q1': 0, 'Q2': 0,
            'Q3': 0, 'Q4': 0
        }
        self.all_latencies = []
        self.rogue_count = 0
        self.throttled_count = 0
        
        self.grid = mesa.space.MultiGrid(
            width, height, torus=False
        )
        
        agents = self._build_population(
            n_agents, queue_scenario
        )
        
        for priority, aggression in agents:
            agent = QueueAgent(self, priority, aggression)
            if agent.is_rogue:
                self.rogue_count += 1
            x = np.random.randint(0, width)
            y = np.random.randint(height // 2, height)
            self.grid.place_agent(agent, (x, y))
    
    def _build_population(self, n_agents, queue_scenario):
        agents = []
        
        if queue_scenario == 'priority_only':
            for i in range(n_agents):
                r = np.random.random()
                if r < 0.10:
                    p = PriorityLevel.CRITICAL
                elif r < 0.25:
                    p = PriorityLevel.HIGH
                elif r < 0.75:
                    p = PriorityLevel.NORMAL
                else:
                    p = PriorityLevel.LOW
                agents.append((p, AggressionLevel.NORMAL))
                
        elif queue_scenario == 'aggression_only':
            for i in range(n_agents):
                r = np.random.random()
                if r < 0.15:
                    a = AggressionLevel.AGGRESSIVE
                elif r < 0.40:
                    a = AggressionLevel.NORMAL
                else:
                    a = AggressionLevel.PASSIVE
                agents.append(
                    (PriorityLevel.NORMAL, a)
                )
                
        elif queue_scenario == 'mixed':
            for i in range(n_agents):
                rp = np.random.random()
                ra = np.random.random()
                
                if rp < 0.10:
                    p = PriorityLevel.CRITICAL
                elif rp < 0.25:
                    p = PriorityLevel.HIGH
                elif rp < 0.75:
                    p = PriorityLevel.NORMAL
                else:
                    p = PriorityLevel.LOW
                    
                if ra < 0.15:
                    a = AggressionLevel.AGGRESSIVE
                elif ra < 0.40:
                    a = AggressionLevel.NORMAL
                else:
                    a = AggressionLevel.PASSIVE
                    
                agents.append((p, a))
        
        return agents
    
    def record_agent(self, agent):
        if agent.latency is not None:
            self.quadrant_latencies[
                agent.quadrant
            ].append(agent.latency)
            self.all_latencies.append(agent.latency)
            self.quadrant_counts[agent.quadrant] += 1
            if agent.was_throttled:
                self.throttled_count += 1
    
    def step(self):
        self.steps += 1
        self.agents.shuffle_do("step")
    
    def run(self, max_steps=500):
        for _ in range(max_steps):
            self.step()
            if self.exited_count >= self.total_agents:
                break
        return self.exited_count, self.exit_times
    
    def results_summary(self):
        summary = {
            'exited': self.exited_count,
            'steps': self.steps,
            'rogue_count': self.rogue_count,
            'throttled_count': self.throttled_count,
            'overall_latency': (
                np.mean(self.all_latencies)
                if self.all_latencies else 0
            ),
            'quadrant_latencies': {}
        }
        
        for q, lats in self.quadrant_latencies.items():
            if lats:
                summary['quadrant_latencies'][q] = {
                    'mean': np.mean(lats),
                    'p95': np.percentile(lats, 95),
                    'count': len(lats)
                }
        
        return summary

print("Two dimensional model defined!")
print("Mesa conflict fixed - using queue_scenario")
print("Scenarios: priority_only, aggression_only, mixed")
print("QoS filter: configurable throttle")
print("Quadrant tracking: Q1, Q2, Q3, Q4")

In [ ]:
exec(open('/home/jc/v7_cell4.py').read())

In [ ]:
print(list(all_results.keys()))
print("Data available for charting")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

scenario_names = list(all_results.keys())
colors = ['steelblue', 'orange', 'red', 'green']

# Chart 1 - Steps to clear
steps = [all_results[s]['avg_steps'] for s in scenario_names]
std = [all_results[s]['std_steps'] for s in scenario_names]
axes[0].bar(scenario_names, steps,
            color=colors, alpha=0.7,
            yerr=std, capsize=5)
axes[0].set_title('Steps to Clear\n(Lower is Better)')
axes[0].set_ylabel('Average Steps')
axes[0].tick_params(axis='x', rotation=15)

# Chart 2 - Overall latency
latencies = [all_results[s]['avg_latency']
             for s in scenario_names]
axes[1].bar(scenario_names, latencies,
            color=colors, alpha=0.7)
axes[1].set_title('Average Latency\n(Lower is Better)')
axes[1].set_ylabel('Average Steps')
axes[1].tick_params(axis='x', rotation=15)

# Chart 3 - Q1 vs Q4 latency
x = np.arange(len(scenario_names))
width = 0.35
q1_lats = [all_results[s]['avg_q1'] for s in scenario_names]
q4_lats = [all_results[s]['avg_q4'] for s in scenario_names]

axes[2].bar(x - width/2, q1_lats, width,
            label='Q1 High P+A', color='green', alpha=0.7)
axes[2].bar(x + width/2, q4_lats, width,
            label='Q4 Rogue', color='red', alpha=0.7)
axes[2].set_title('Q1 vs Q4 Latency\n(Fairness)')
axes[2].set_ylabel('Average Latency')
axes[2].set_xticks(x)
axes[2].set_xticklabels(scenario_names, rotation=15)
axes[2].legend(fontsize=8)

plt.suptitle(
    'V7 Two Dimensional Queue Model\nPriority vs Aggression vs QoS',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig('/home/jc/v7_comparison.png', dpi=150)
plt.show()
print("Chart saved!")

In [ ]:
# 5000 AGENT SCALE TEST - V7 Two Dimensional Model
# Testing all four scenarios at large scale
print("V7 LARGE SCALE TEST - 5000 AGENTS")
print("="*55)
print("Running all four scenarios at large scale...")
print("This will take several hours - starting now")
print("="*55)

scale_5k_scenarios = [
    {'name': 'Priority Only', 
     'queue_scenario': 'priority_only', 
     'qos_enabled': False},
    {'name': 'Aggression Only', 
     'queue_scenario': 'aggression_only', 
     'qos_enabled': False},
    {'name': 'Mixed No QoS', 
     'queue_scenario': 'mixed', 
     'qos_enabled': False},
    {'name': 'Mixed With QoS', 
     'queue_scenario': 'mixed', 
     'qos_enabled': True}
]

n_runs = 20
results_5k = {}

for scenario in scale_5k_scenarios:
    print(f"\nScenario: {scenario['name']}")
    print("-"*55)
    
    run_steps = []
    run_latencies = []
    run_q1_lat = []
    run_q2_lat = []
    run_q3_lat = []
    run_q4_lat = []
    run_throttled = []
    run_rogue = []
    
    for run in range(n_runs):
        model = BunchQueueModel(
            n_agents=5000,
            width=140,
            height=200,
            queue_scenario=scenario['queue_scenario'],
            qos_enabled=scenario['qos_enabled'],
            qos_throttle=0.3
        )
        model.run(max_steps=2000)
        summary = model.results_summary()
        
        run_steps.append(summary['steps'])
        run_latencies.append(summary['overall_latency'])
        run_throttled.append(summary['throttled_count'])
        run_rogue.append(summary['rogue_count'])
        ql = summary['quadrant_latencies']
        if 'Q1' in ql: run_q1_lat.append(ql['Q1']['mean'])
        if 'Q2' in ql: run_q2_lat.append(ql['Q2']['mean'])
        if 'Q3' in ql: run_q3_lat.append(ql['Q3']['mean'])
        if 'Q4' in ql: run_q4_lat.append(ql['Q4']['mean'])
        
        print(f"  Run {run+1:2d}/{n_runs} — "
              f"steps: {summary['steps']} "
              f"latency: {summary['overall_latency']:.1f}")
    
    results_5k[scenario['name']] = {
        'avg_steps': np.mean(run_steps),
        'std_steps': np.std(run_steps),
        'avg_latency': np.mean(run_latencies),
        'avg_q1': np.mean(run_q1_lat) if run_q1_lat else 0,
        'avg_q2': np.mean(run_q2_lat) if run_q2_lat else 0,
        'avg_q3': np.mean(run_q3_lat) if run_q3_lat else 0,
        'avg_q4': np.mean(run_q4_lat) if run_q4_lat else 0,
        'avg_throttled': np.mean(run_throttled),
        'avg_rogue': np.mean(run_rogue)
    }
    
    r = results_5k[scenario['name']]
    print(f"\n  COMPLETED: {scenario['name']}")
    print(f"  Avg steps:   {r['avg_steps']:.1f} "
          f"+/- {r['std_steps']:.1f}")
    print(f"  Avg latency: {r['avg_latency']:.1f}")
    print(f"  Q1:{r['avg_q1']:.1f} Q2:{r['avg_q2']:.1f} "
          f"Q3:{r['avg_q3']:.1f} Q4:{r['avg_q4']:.1f}")

print("\n" + "="*55)
print("5000 AGENT SCALE TEST COMPLETE")
print("="*55)
print(f"{'Scenario':<20} {'Steps':<10} {'Latency':<10} "
      f"{'Q1':<8} {'Q4':<8} {'Fair?'}")
print("-"*55)

for name, r in results_5k.items():
    if r['avg_q1'] > 0 and r['avg_q4'] > 0:
        fair = "YES" if r['avg_q1'] < r['avg_q4'] else "NO"
    else:
        fair = "N/A"
    print(f"{name:<20} {r['avg_steps']:<10.1f} "
          f"{r['avg_latency']:<10.1f} "
          f"{r['avg_q1']:<8.1f} "
          f"{r['avg_q4']:<8.1f} {fair}")
print("="*55)